In [149]:
# Core imports
import sys
import json
import os
from pathlib import Path

import pandas as pd

_here = Path.cwd()
if (_here / "utils").exists():
    pass
elif (_here / "notebooks" / "utils").exists():
    sys.path.insert(0, str(_here / "notebooks"))

import dspy
from typing import List, Optional, Literal, Union
from pydantic import BaseModel, Field
from tqdm import tqdm


In [150]:
# DSPy Configuration
lm = dspy.LM(
    model="openai/gpt-5.2", 
    api_key=os.environ["OPENAI_API_KEY"],
)
dspy.configure(lm=lm)



### Core Policy Model

In [151]:
from utils.schemas import ExtractedPolicy

In [152]:
from utils.schemas import DocumentMetadata 


---

## 🧠 DSPy Signatures & Modules {#dspy}

Define DSPy signatures and custom modules for policy extraction, validation, and processing.

### Policy Extraction Signature

In [153]:
from utils.dspy_extraction import PolicyExtractionSignature, PolicyExtractor

In [154]:
# (Legacy strict validator removed)
# Individual validation uses the refined validator in `utils.dspy_validation`.
from utils.dspy_validation import PolicyValidationSignature, PolicyValidator, ValidationMetrics

In [155]:
from utils.docling_io import pdf_to_markdown

---

## 🗂️ Batch Configuration

Define the list of cities to process. Add a `DocumentMetadata` + PDF path per city. All 7 steps below will loop over this list automatically.

In [156]:
# ─── Batch Configuration ────────────────────────────────────────────────────
# Each entry has:
#   metadata     -> DocumentMetadata (country, state_or_province, city)
#   pdf_path     -> original PDF (only used if markdown_path is missing)
#   markdown_path -> pre-converted markdown file (skips PDF conversion entirely)
#
# Outputs are written to OUTPUT_DIR/<city_key>/

from utils.schemas import DocumentMetadata

BATCH = [
    {
        "metadata": DocumentMetadata(country="United States", state_or_province="Illinois", city="Chicago"),
        "pdf_path": "../pdfs/Chicago-CAP-071822.pdf",
        "markdown_path": "../docs/cities/chicago.md",
    },
    {
        "metadata": DocumentMetadata(country="United States", state_or_province="Washington", city="Seattle"),
        "pdf_path": "../pdfs/Seattle.pdf",
        "markdown_path": "../docs/cities/seattle_markdown.md",
    },
    {
        "metadata": DocumentMetadata(country="United States", state_or_province="Nevada", city="Las Vegas"),
        "pdf_path": "../pdfs/CLV.pdf",
        "markdown_path": "../docs/cities/LV.md",
    },
    {
        "metadata": DocumentMetadata(country="United States", state_or_province="Florida", city="Miami-Dade"),
        "pdf_path": "../pdfs/MIAMI_DADE.pdf",
        "markdown_path": "../docs/cities/miami_markdown.md",
    },
    {
        "metadata": DocumentMetadata(country="United States", state_or_province="Texas", city="Austin"),
        "pdf_path": "../pdfs/Austin_climate_equity.pdf",
        "markdown_path": "../docs/cities/austin.md",
    },
    {
        "metadata": DocumentMetadata(country="Senegal", city="Dakar"),
        "pdf_path": "../pdfs/Dakar.pdf",
        "markdown_path": "../docs/cities/dakar.md",
    },
    {
        "metadata": DocumentMetadata(country="Kuwait"),
        "pdf_path": "../pdfs/Kuwait-NAP-2019-2030.pdf",
        "markdown_path": "../docs/cities/kuwait.md",
    },
    {
        "metadata": DocumentMetadata(country="Portugal"),
        "pdf_path": "../pdfs/2021 Portugal ADCOM_UNFCCC.pdf",
        "markdown_path": "../docs/cities/Portugal.md",
    },
    {
        "metadata": DocumentMetadata(country = "Switzerland",city= "Geneva"),
        "pdf_path":"../pdfs/city_of_geneva.pdf",
        "markdown_path":"../docs/cities/geneva.md",

    },
    {
        "metadata":DocumentMetadata(country="Japan",city="Hiroshima"),
        "pdf_path":"../pdfs/HIROSHIMA.pdf",
        "markdown_path":"../docs/cities/Hiroshima.md",
    }
]

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Number of parallel threads for LLM calls (Steps 5 & 6).
NUM_THREADS = 8

def city_key(metadata: DocumentMetadata) -> str:
    """Slug used as filename prefix and output folder name."""
    parts = [metadata.city or metadata.country, metadata.country]
    # e.g. 'Chicago_United_States' or 'Kuwait_Kuwait' → just use city if present
    label = metadata.city if metadata.city else metadata.country
    return label.replace(" ", "_").replace("-", "_")

print(f"Batch configured ({len(BATCH)} cities):")
for e in BATCH:
    key = city_key(e["metadata"])
    md = e.get("markdown_path", "—")
    print(f"  {key:20s}  md={md}")
print(f"\nParallel threads: {NUM_THREADS}")


Batch configured (10 cities):
  Chicago               md=../docs/cities/chicago.md
  Seattle               md=../docs/cities/seattle_markdown.md
  Las_Vegas             md=../docs/cities/LV.md
  Miami_Dade            md=../docs/cities/miami_markdown.md
  Austin                md=../docs/cities/austin.md
  Dakar                 md=../docs/cities/dakar.md
  Kuwait                md=../docs/cities/kuwait.md
  Portugal              md=../docs/cities/Portugal.md
  Geneva                md=../docs/cities/geneva.md
  Hiroshima             md=../docs/cities/Hiroshima.md

Parallel threads: 8


### Step 1 — Load Markdown

Loads each city's document as markdown text using this priority order:
1. **Pre-converted file** supplied via `markdown_path` in `BATCH` (e.g. `docs/cities/*.md`) — instant, no API call
2. **Cached conversion** saved by a previous run (`outputs/{key}_markdown.md`)
3. **Live PDF conversion** via Docling — only runs if neither of the above exists, and caches the result


In [157]:
# Step 1: Load Markdown (pre-converted files take priority; falls back to PDF conversion)
from utils.docling_io import pdf_to_markdown

markdowns = {}  # city_key -> markdown string

for entry in BATCH:
    key = city_key(entry["metadata"])
    md_path = entry.get("markdown_path")

    # ── Priority 1: pre-existing markdown file supplied in config ────────
    if md_path and Path(md_path).exists():
        with open(md_path, "r", encoding="utf-8") as f:
            markdowns[key] = f.read()
        print(f"[{key}] Loaded pre-converted markdown from {md_path}  ({len(markdowns[key]):,} chars)")

    # ── Priority 2: cached conversion saved by a previous run ────────────
    elif (OUTPUT_DIR / f"{key}_markdown.md").exists():
        cached = OUTPUT_DIR / f"{key}_markdown.md"
        with open(cached, "r", encoding="utf-8") as f:
            markdowns[key] = f.read()
        print(f"[{key}] Loaded cached markdown from {cached}  ({len(markdowns[key]):,} chars)")

    # ── Priority 3: convert from PDF and cache the result ────────────────
    else:
        pdf = entry.get("pdf_path")
        if not pdf or not Path(pdf).exists():
            print(f"[{key}]  No markdown and no PDF found — skipping.")
            continue
        cache_path = OUTPUT_DIR / f"{key}_markdown.md"
        print(f"[{key}] Converting PDF: {pdf} ...")
        markdowns[key] = pdf_to_markdown(pdf, save_markdown_path=str(cache_path))
        print(f"[{key}] Done  ({len(markdowns[key]):,} chars)  → cached at {cache_path}")


[Chicago] Loaded pre-converted markdown from ../docs/cities/chicago.md  (256,905 chars)
[Seattle] Loaded pre-converted markdown from ../docs/cities/seattle_markdown.md  (49,888 chars)
[Las_Vegas] Loaded pre-converted markdown from ../docs/cities/LV.md  (61,953 chars)
[Miami_Dade] Loaded pre-converted markdown from ../docs/cities/miami_markdown.md  (141,191 chars)
[Austin] Loaded pre-converted markdown from ../docs/cities/austin.md  (413,340 chars)
[Dakar] Loaded pre-converted markdown from ../docs/cities/dakar.md  (227,420 chars)
[Kuwait] Loaded pre-converted markdown from ../docs/cities/kuwait.md  (362,332 chars)
[Portugal] Loaded pre-converted markdown from ../docs/cities/Portugal.md  (438,819 chars)
[Geneva] Loaded pre-converted markdown from ../docs/cities/geneva.md  (19,174 chars)
[Hiroshima] Loaded pre-converted markdown from ../docs/cities/Hiroshima.md  (87,414 chars)


### Step 2 — Chunk & Extract Policies

Chunks each city's document using `DocumentChunker` (heading-aware greedy bin-packing, 6 000-word budget, 2-paragraph overlap window), then runs `PolicyExtractor` per chunk with a carry-forward summary for cross-chunk context. All chunks go through `PolicyResolver` (dedup → deterministic link → LLM arbitration) before results are stored.

Saves `{city_key}_extracted_policies.json` to `outputs/`.

In [158]:
# Step 2: Chunk, Extract & Resolve — ExtractionPipeline handles chunking,
# carry-forward summary, per-chunk retry, and three-stage resolution.

from utils.dspy_extraction import PolicyExtractor
from utils.chunking import DocumentChunker
from utils.dspy_resolve import PolicyResolver
from utils.pipeline import ExtractionPipeline
from utils.schemas import make_doc_id

policy_extractor = PolicyExtractor()
chunker  = DocumentChunker(words_per_chunk=37500, overlap_paragraphs=2)
resolver = PolicyResolver()
pipeline = ExtractionPipeline(
    extractor=policy_extractor,
    chunker=chunker,
    resolver=resolver,
    max_chunk_retries=1,
)

all_extracted = {}      # city_key -> List[ExtractedPolicy]
pipeline_results = {}   # city_key -> PipelineResult (for chunk traces)

batch_to_extract = [e for e in BATCH if city_key(e["metadata"]) in markdowns]
print(f"Extracting policies for {len(batch_to_extract)} cities...\n")

for entry in batch_to_extract:
    key  = city_key(entry["metadata"])
    meta = entry["metadata"]
    text = markdowns[key]

    # Stamp a stable doc_id on the metadata before extraction
    meta.doc_id = make_doc_id(meta, text)
    print(f"\n[{key}]  doc_id={meta.doc_id}")

    result = pipeline.run(text, meta)
    all_extracted[key]    = result.policies
    pipeline_results[key] = result

    # Write extracted policies
    out_json = OUTPUT_DIR / f"{key}_extracted_policies.json"
    with open(out_json, "w", encoding="utf-8") as f:
        import json as _json
        _json.dump(
            [p.model_dump() for p in result.policies],
            f, ensure_ascii=False, indent=2,
        )

    # Write chunk trace
    trace_path = ExtractionPipeline.export_chunk_trace(result, OUTPUT_DIR / key)

    print(f"  {len(result.policies)} policies after resolve  "
          f"({result.total_chunks} chunk(s))  → {out_json}")
    print(f"  chunk trace → {trace_path}")

print(f"\nDone. {len(all_extracted)} cities processed.")

Extracting policies for 10 cities...


[Chicago]  doc_id=Chicago_Illinois_United_States_b777fa82c6
  [Chicago_Illinois_United_States_b777fa82c6] 1 chunk(s), 34077 total words
    chunk 1/1: 92 policies (0.1s)
    [Chicago_Illinois_United_States_b777fa82c6] dedup: 1 verbatim, 2 statement
    [Chicago_Illinois_United_States_b777fa82c6] linked: 48 deterministic
  89 policies after resolve  (1 chunk(s))  → outputs/Chicago_extracted_policies.json
  chunk trace → outputs/Chicago/chunk_trace.json

[Seattle]  doc_id=Seattle_Washington_United_States_0ff1537e21
  [Seattle_Washington_United_States_0ff1537e21] 1 chunk(s), 6822 total words
    chunk 1/1: 30 policies (0.1s)
    [Seattle_Washington_United_States_0ff1537e21] linked: 12 deterministic
  30 policies after resolve  (1 chunk(s))  → outputs/Seattle_extracted_policies.json
  chunk trace → outputs/Seattle/chunk_trace.json

[Las_Vegas]  doc_id=Las_Vegas_Nevada_United_States_87cbfaf27b
  [Las_Vegas_Nevada_United_States_87cbfaf27b] 1 chunk(s), 9

### Step 3 — Cluster Policies

Groups policies using the resolver's `parent_policy_name` linkage (set in Step 2) into `parent_with_subs`, `individual`, and `orphan_sub` clusters. Each cluster gets a stable `cluster_id`. Saves `{city_key}_policy_clusters.json`.

In [159]:
# Step 3: Cluster Policies
from utils.clustering import cluster_policies, summarize_clusters

all_clusters = {}  # city_key -> List[dict]

for entry in BATCH:
    key = city_key(entry["metadata"])
    clusters = cluster_policies(all_extracted[key])
    all_clusters[key] = clusters

    print(f"\n[{key}]")
    summarize_clusters(clusters)

    out = OUTPUT_DIR / f"{key}_policy_clusters.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(clusters, f, default=lambda o: o.model_dump(), ensure_ascii=False, indent=2)
    print(f"  → {out}")



[Chicago]
Total clusters:        41
  Parent+sub clusters: 19
  Individual clusters: 22
  Orphaned subs:       0

[Parent 1] [climate action strategies]
  Pillar 1: Increase access to utility savings and renewable energy, prioritizing 
[Parent 2] [pillar 1 | strategy 1. retrofit buildings]
  Strategy 1 (Pillar 1): Retrofit buildings.
  └─ Sub 1: Retrofit residential buildings with 4 or fewer units: retrofit 20% by 
  └─ Sub 2: Retrofit 20% of total 5+ unit residential buildings by 2030.
  └─ Sub 3: Retrofit 20% of total industrial buildings by 2030.
  └─ Sub 4: Retrofit 90% of total City-owned and sister agency-owned buildings by 
[Parent 3] [pillar 1 | strategy 2. connect communities to renewable energy]
  Strategy 2 (Pillar 1): Connect communities to renewable energy.
  └─ Sub 1: Install 5 megawatts of co-owned community solar projects by 2025.
  └─ Sub 2: Increase Chicago-based community renewables to 20 megawatts by 2025.
  └─ Sub 3: Increase community renewables subscriptions so 

### Step 4 — Build Policy Records

Flattens each city's clusters into a structured DataFrame with consistent columns. Saves `{City_Country}_policy_records.csv`.

In [160]:
# Step 4: Build Policy Records DataFrames
from utils.clustering import clusters_to_records

FRONT_COLS = [
    "cluster_id", "cluster_type", "role", "section_header", "sector",
    "policy_statement", "parent_statement", "verbatim_text", "extraction_rationale",
]

all_policy_records = {}  # city_key -> List[dict]
all_df_policies = {}     # city_key -> pd.DataFrame

for entry in BATCH:
    key = city_key(entry["metadata"])
    records = clusters_to_records(all_clusters[key])
    df = pd.DataFrame(records)
    df = df[FRONT_COLS + [c for c in df.columns if c not in FRONT_COLS]]

    all_policy_records[key] = records
    all_df_policies[key] = df

    # out = OUTPUT_DIR / f"{key}_policy_records.csv"
    # df.to_csv(out, index=False)
    # print(f"[{key}] {len(df)} records → {out}")


### Step 5 — Validate Individual Policies

Runs `PolicyValidator` on each `individual`-role policy. Flattens Pydantic metrics into columns. Saves `{City_Country}_validation_individual.csv`.

In [161]:
# Step 5: Validate Individual Policies — parallel LLM calls via DSPy ParallelExecutor
from utils.dspy_validation import PolicyValidator
from dspy.utils.parallelizer import ParallelExecutor

validator = PolicyValidator()
all_df_final = {}  # city_key -> pd.DataFrame (flattened metrics)

for entry in BATCH:
    key = city_key(entry["metadata"])
    individual = [r for r in all_policy_records[key] if r.get("role") == "individual"]
    print(f"\n[{key}] Validating {len(individual)} individual policies "
          f"across {NUM_THREADS} threads...")

    # ── Build per-record task function ──────────────────────────────────
    def validate_one(record):
        pred = validator(policy_data=record)
        return {**record, **pred.toDict()}

    # ── Execute in parallel — results land in-order, errors → None ──────
    executor = ParallelExecutor(
        num_threads=NUM_THREADS,
        max_errors=len(individual),   # don't abort on individual failures
        provide_traceback=True,
    )
    raw_results = executor.execute(validate_one, individual)

    # ── Filter out None (failed calls) and collect errors ────────────────
    validated = []
    n_errors = 0
    for i, res in enumerate(raw_results):
        if res is None or isinstance(res, Exception):
            n_errors += 1
            print(f"  ⚠ Record {i} failed: {res}")
        else:
            validated.append(res)

    print(f"[{key}] {len(validated)} succeeded, {n_errors} failed")

    # ── Build DataFrame (all results now in memory) ───────────────────────
    df_val = pd.DataFrame(validated)

    # Flatten Pydantic validation_results column into separate columns
    if "validation_results" in df_val.columns:
        vdata = df_val["validation_results"].apply(
            lambda x: x.model_dump() if hasattr(x, "model_dump") else x.dict()
        )
        df_metrics = pd.json_normalize(vdata)
        df_val = pd.concat([df_val.drop(columns=["validation_results"]), df_metrics], axis=1)

    all_df_final[key] = df_val

    # ── Write once, after all results are collected ───────────────────────
    # out = OUTPUT_DIR / f"{key}_validation_individual.csv"
    # df_val.to_csv(out, index=False)
    # print(f"[{key}] Saved → {out}")



[Chicago] Validating 22 individual policies across 8 threads...
Processed 22 / 22 examples: 100%|██████████| 22/22 [00:00<00:00, 42.06it/s]
[Chicago] 22 succeeded, 0 failed

[Seattle] Validating 15 individual policies across 8 threads...
Processed 15 / 15 examples: 100%|██████████| 15/15 [00:00<00:00, 79.25it/s]
[Seattle] 15 succeeded, 0 failed

[Las_Vegas] Validating 12 individual policies across 8 threads...
Processed 12 / 12 examples: 100%|██████████| 12/12 [00:00<00:00, 77.27it/s]
[Las_Vegas] 12 succeeded, 0 failed

[Miami_Dade] Validating 5 individual policies across 8 threads...
Processed 5 / 5 examples: 100%|██████████| 5/5 [00:00<00:00, 89.91it/s]
[Miami_Dade] 5 succeeded, 0 failed

[Austin] Validating 2 individual policies across 8 threads...
Processed 2 / 2 examples: 100%|██████████| 2/2 [00:00<00:00, 88.94it/s]
[Austin] 2 succeeded, 0 failed

[Dakar] Validating 13 individual policies across 8 threads...
Processed 13 / 13 examples: 100%|██████████| 13/13 [00:00<00:00, 113.28

### Step 6 — Validate Initiatives

Runs `run_initiative_validation` on each city's `parent_with_subs` clusters. Saves `{City_Country}_validated_initiatives.csv`. 


In [162]:
# Step 6: Validate Initiatives — parallel LLM calls via DSPy ParallelExecutor
import importlib

from dspy.utils.parallelizer import ParallelExecutor
from utils import initiative_validator as initiative_validator_mod

initiative_validator_mod = importlib.reload(initiative_validator_mod)
InitiativeValidator = initiative_validator_mod.InitiativeValidator
build_initiative_context = initiative_validator_mod.build_initiative_context
compute_initiative_final_verdict = initiative_validator_mod.compute_initiative_final_verdict

init_validator = InitiativeValidator()
all_df_initiatives = {}  # city_key -> pd.DataFrame

for entry in BATCH:
    key = city_key(entry["metadata"])
    initiative_clusters = [
        c for c in all_clusters[key] if c["cluster_type"] == "parent_with_subs"
    ]
    if not initiative_clusters:
        print(f"No parent_with_subs clusters — skipping.")
        all_df_initiatives[key] = pd.DataFrame()
        continue

    # Per-cluster task
    def validate_initiative(cluster):
        context = build_initiative_context(cluster)
        prediction = init_validator(initiative_data=context)
        metrics = prediction.initiative_metrics
        return {
            "initiative_name":              context["initiative_name"],
            "parent_statement":             context["parent_statement"],
            "sector":                       context["sector"],
            "num_sub_actions":              context["num_sub_actions"],
            "coverage_score":               metrics.coverage_score,
            "coverage_assessment":          metrics.coverage_assessment,
            "coherence_score":              metrics.coherence_score,
            "coherence_assessment":         metrics.coherence_assessment,
            "aggregate_measurability":      metrics.aggregate_measurability,
            "has_implementation_pathway":   metrics.has_implementation_pathway,
            "inherited_binding_mechanism":  metrics.inherited_binding_mechanism,
            "inherited_spatial_scope":      metrics.inherited_spatial_scope,
            "weak_signals":                 metrics.weak_signals,
            "strong_signals":               metrics.strong_signals,
            "initiative_result":            metrics.initiative_result,
            "confidence_score":             metrics.confidence_score,
            "initiative_reasoning":         metrics.initiative_reasoning,
            "final_verdict":                compute_initiative_final_verdict(
                metrics.initiative_result,
                metrics.confidence_score,
            ),
            "parent_row_type":              metrics.parent_row_type,
            "parent_row_type_reasoning":    metrics.parent_row_type_reasoning,
            "sub_assessments": [
                sa.model_dump() if hasattr(sa, "model_dump") else sa.dict()
                for sa in metrics.sub_assessments
            ],
            "subs_strong":   sum(1 for sa in metrics.sub_assessments if sa.strength == "strong"),
            "subs_moderate": sum(1 for sa in metrics.sub_assessments if sa.strength == "moderate"),
            "subs_weak":     sum(1 for sa in metrics.sub_assessments if sa.strength == "weak"),
        }

    # Execute in parallel
    executor = ParallelExecutor(
        num_threads=NUM_THREADS,
        max_errors=len(initiative_clusters),
        provide_traceback=True,
    )
    raw_results = executor.execute(validate_initiative, initiative_clusters)

    # Filter failures
    results, n_errors = [], 0
    for i, res in enumerate(raw_results):
        if res is None or isinstance(res, Exception):
            n_errors += 1
            print(f"Cluster {i} failed: {res}")
        else:
            results.append(res)

    print(f"[{key}] {len(results)} succeeded, {n_errors} failed")

    # Build DataFrame + print summary
    df_inits = pd.DataFrame(results)
    if not df_inits.empty:
        print(f"  SOUND:   {(df_inits['initiative_result'] == 'SOUND').sum()}")
        print(f"  PARTIAL: {(df_inits['initiative_result'] == 'PARTIAL').sum()}")
        print(f"  WEAK:    {(df_inits['initiative_result'] == 'WEAK').sum()}")
        print(f"  final_verdict=True: {df_inits['final_verdict'].sum()}")
        if "parent_row_type" in df_inits.columns:
            print(f"  parent_row_type filled: {df_inits['parent_row_type'].notna().sum()}/{len(df_inits)}")

    all_df_initiatives[key] = df_inits

    # Write once, after all results are collected. initiatives are built
    # out = OUTPUT_DIR / f"{key}_validation_initiatives.csv"
    # df_inits.to_csv(out, index=False)
    # print(f"[{key}] Saved → {out}")


AttributeError: module 'utils.initiative_validator' has no attribute 'compute_initiative_final_verdict'

### Step 7 — Export Combined Results

Builds the final combined table per city and writes `combined_policies.csv`, `trace_individual_policies.csv`, and `trace_initiative_policies.csv` under `outputs/{City_Country}/`.

In [ ]:
# Step 7: Export Combined Results per City
import importlib

from utils import exports as exports_mod

exports_mod = importlib.reload(exports_mod)
build_combined_policies_table = exports_mod.build_combined_policies_table
export_combined_table_and_traces = exports_mod.export_combined_table_and_traces
filter_valid_policies = exports_mod.filter_valid_policies

all_combined = {}  # city_key -> pd.DataFrame (all policies)
all_valid    = {}  # city_key -> pd.DataFrame (final_verdict == True only)

for entry in BATCH:
    key = city_key(entry["metadata"])
    city_output_dir = OUTPUT_DIR / key
    df_inits = all_df_initiatives.get(key, pd.DataFrame())
    df_inits_arg = df_inits if not df_inits.empty else None

    combined = build_combined_policies_table(
        df_policies=all_df_policies[key],
        df_final_individual=all_df_final[key],
        df_initiatives=df_inits,
        policy_clusters=all_clusters[key],
    )
    all_combined[key] = combined

    # combined_policies.csv now contains valid policies only (individual + initiative clusters)
    # traces contain everything (all rows, full validation detail)
    paths = export_combined_table_and_traces(
        combined=combined,
        df_initiatives=df_inits_arg,
        df_final_individual=all_df_final[key],
        output_dir=city_output_dir,
    )

    # Keep an in-memory copy of valid policies (final_verdict == True)
    df_valid = filter_valid_policies(combined, all_df_final[key], df_inits_arg)
    all_valid[key] = df_valid

    n_all   = len(combined)
    n_valid = len(df_valid)
    print(f"\n[{key}] {n_valid}/{n_all} valid policies  →  {city_output_dir}/combined_policies.csv")
    if df_inits_arg is not None and "parent_row_type" in df_inits_arg.columns:
        print(f"  initiative parent_row_type filled: {df_inits_arg['parent_row_type'].notna().sum()}/{len(df_inits_arg)}")
    if "parent_row_type" in df_valid.columns:
        print(f"  valid rows with parent_row_type: {df_valid['parent_row_type'].notna().sum()}/{len(df_valid)}")
    for k, v in paths.items():
        print(f"  {k}: {v}")



[Chicago] 62/89 valid policies  →  outputs/Chicago/combined_policies.csv
  valid rows with parent_row_type: 0/62
  combined_policies: outputs/Chicago/combined_policies.csv
  trace_individual: outputs/Chicago/trace_individual_policies.csv
  trace_individual_valid: outputs/Chicago/trace_individual_policies_valid.csv
  trace_initiatives: outputs/Chicago/trace_initiative_policies.csv
  trace_initiatives_valid: outputs/Chicago/trace_initiative_policies_valid.csv

[Seattle] 17/30 valid policies  →  outputs/Seattle/combined_policies.csv
  valid rows with parent_row_type: 0/17
  combined_policies: outputs/Seattle/combined_policies.csv
  trace_individual: outputs/Seattle/trace_individual_policies.csv
  trace_individual_valid: outputs/Seattle/trace_individual_policies_valid.csv
  trace_initiatives: outputs/Seattle/trace_initiative_policies.csv
  trace_initiatives_valid: outputs/Seattle/trace_initiative_policies_valid.csv

[Las_Vegas] 16/57 valid policies  →  outputs/Las_Vegas/combined_policies.

In [ ]:
#

In [ ]:

# # ── Load all_valid from disk for Step 8 (skips Steps 1–7) ────────────────────
# #
# # CLASSIFICATION_INPUT controls what gets fed into Step 8:
# #
# #   "valid_only"    — reads combined_policies.csv (individual + parent + sub rows
# #                     where final_verdict == True).  Fast; uses the saved output.
# #                     ⚠ Cities with no passing initiatives will only have their
# #                     individually-validated policies (e.g. Seattle = 5 rows).
# #
# #   "all_extracted" — re-clusters the cached extracted_policies.json (no LLM calls),
# #                     then keeps ALL individual + sub policies regardless of verdict.
# #                     Use this when you want to classify every extracted policy,
# #                     including sub-actions from initiatives that failed validation.
# #
# CLASSIFICATION_INPUT = "valid_only"   # "valid_only" | "all_extracted"

# # ─────────────────────────────────────────────────────────────────────────────
# OUTPUT_DIR = Path("outputs")

# try:
#     BATCH
# except NameError:
#     from utils.schemas import DocumentMetadata
#     BATCH = [
#         {"metadata": DocumentMetadata(country="United States", state_or_province="Illinois",  city="Chicago")},
#         {"metadata": DocumentMetadata(country="United States", state_or_province="Washington", city="Seattle")},
#         {"metadata": DocumentMetadata(country="United States", state_or_province="Nevada",     city="Las Vegas")},
#         {"metadata": DocumentMetadata(country="United States", state_or_province="Florida",    city="Miami-Dade")},
#         {"metadata": DocumentMetadata(country="United States", state_or_province="Texas",      city="Austin")},
#         {"metadata": DocumentMetadata(country="Senegal",                                       city="Dakar")},
#         {"metadata": DocumentMetadata(country="Kuwait")},
#         {"metadata": DocumentMetadata(country="Portugal")},
#         {"metadata": DocumentMetadata(country="Switzerland",                                   city="Geneva")},
#         {"metadata": DocumentMetadata(country="Japan",                                         city="Hiroshima")},
#     ]

#     def city_key(metadata):
#         label = metadata.city if metadata.city else metadata.country
#         return label.replace(" ", "_").replace("-", "_")

# all_valid = {}

# if CLASSIFICATION_INPUT == "valid_only":
#     # ── Mode 1: read combined_policies.csv (valid rows only) ─────────────────
#     for entry in BATCH:
#         key = city_key(entry["metadata"])
#         csv_path = OUTPUT_DIR / key / "combined_policies.csv"
#         if csv_path.exists():
#             df = pd.read_csv(csv_path)
#             all_valid[key] = df
#             role_dist = df["role"].value_counts().to_dict() if "role" in df.columns else {}
#             print(f"[{key}] {len(df)} valid policies — {role_dist}")
#         else:
#             print(f"[{key}] ⚠ {csv_path} not found — skipping.")

# elif CLASSIFICATION_INPUT == "all_extracted":
#     # ── Mode 2: include ALL individual + sub rows without LLM calls ────────────
#     # Preferred source: cached extracted JSON (lets us re-run clustering).
#     # Fallback source: existing CSV outputs under outputs/{City}/.
#     from utils.clustering import cluster_policies, clusters_to_records
#     from utils.schemas import ExtractedPolicy
#     import ast

#     def _ensure_cols(df: pd.DataFrame) -> pd.DataFrame:
#         for col in (
#             "verbatim_text", "policy_statement", "sector", "section_header",
#             "parent_statement", "cluster_id", "cluster_type", "role",
#         ):
#             if col not in df.columns:
#                 df[col] = None
#         return df

#     for entry in BATCH:
#         key = city_key(entry["metadata"])

#         # 1) Best: cached extracted JSON (flat)
#         candidate_jsons = [
#             OUTPUT_DIR / f"{key}_extracted_policies.json",
#             OUTPUT_DIR / key / "extracted_policies.json",
#             OUTPUT_DIR / key / f"{key}_extracted_policies.json",
#         ]
#         json_path = next((p for p in candidate_jsons if p.exists()), None)

#         if json_path is not None:
#             with open(json_path, "r", encoding="utf-8") as f:
#                 raw = json.load(f)

#             extracted = [ExtractedPolicy(**p) for p in raw]
#             clusters  = cluster_policies(extracted)
#             records   = clusters_to_records(clusters)

#             df = pd.DataFrame([
#                 r for r in records
#                 if r.get("role") in ("individual", "sub", "orphan_sub")
#             ])
#             if df.empty:
#                 print(f"[{key}] ⚠ No individual/sub policies found after clustering from {json_path}.")
#                 continue

#             df = _ensure_cols(df)
#             all_valid[key] = df
#             role_dist = df["role"].value_counts().to_dict()
#             print(f"[{key}] {len(df)} policies (all extracted) — {role_dist}  [source: {json_path}]")
#             continue

#         # 2) Fallback: build a classification input from existing CSV outputs
#         city_dir = OUTPUT_DIR / key
#         combined_path = city_dir / "combined_policies.csv"
#         ind_trace_path = city_dir / "trace_individual_policies.csv"
#         init_trace_path = city_dir / "trace_initiative_policies.csv"

#         parts = []

#         # - subs + some individual rows (usually VALID only)
#         if combined_path.exists():
#             df_c = pd.read_csv(combined_path)
#             if "role" not in df_c.columns:
#                 df_c["role"] = None
#             df_c = df_c[df_c["role"].isin(["individual", "sub", "orphan_sub"])].copy()
#             parts.append(df_c)

#         # - individual trace (includes VALID + INVALID individuals)
#         if ind_trace_path.exists():
#             df_i = pd.read_csv(ind_trace_path)
#             if "role" not in df_i.columns:
#                 df_i["role"] = "individual"
#             df_i = df_i[df_i["role"].isin(["individual"])].copy()
#             parts.append(df_i)

#         # - initiative trace: expand sub_assessments into synthetic sub rows (best-effort)
#         if init_trace_path.exists():
#             df_init = pd.read_csv(init_trace_path)
#             synth = []
#             for _, row in df_init.iterrows():
#                 raw_subs = row.get("sub_assessments")
#                 if not isinstance(raw_subs, str) or not raw_subs.strip():
#                     continue
#                 try:
#                     subs = ast.literal_eval(raw_subs)
#                 except Exception:
#                     continue
#                 if not isinstance(subs, list):
#                     continue
#                 for s in subs:
#                     if not isinstance(s, dict):
#                         continue
#                     stmt = s.get("action_label") or s.get("action")
#                     if not stmt:
#                         continue
#                     synth.append({
#                         "role": "sub",
#                         "cluster_type": "parent_with_subs",
#                         "cluster_id": None,
#                         "sector": row.get("sector"),
#                         "section_header": row.get("initiative_name"),
#                         "parent_statement": row.get("parent_statement"),
#                         "policy_statement": stmt,
#                         "verbatim_text": stmt,
#                     })

#             if synth:
#                 parts.append(pd.DataFrame(synth))

#         if not parts:
#             print(f"[{key}] ⚠ No cached JSON and no usable CSV outputs under {city_dir} — skipping.")
#             continue

#         df = pd.concat(parts, ignore_index=True, sort=False)
#         df = _ensure_cols(df)

#         # keep only rows we can classify
#         df = df[df["role"].isin(["individual", "sub", "orphan_sub"])].copy()
#         df = df[df["policy_statement"].notna() & (df["policy_statement"].astype(str).str.strip() != "")]

#         if df.empty:
#             print(f"[{key}] ⚠ Fallback CSV sources produced 0 classify-able rows — skipping.")
#             continue

#         all_valid[key] = df
#         role_dist = df["role"].value_counts().to_dict()
#         print(f"[{key}] {len(df)} policies (all extracted, fallback) — {role_dist}  [source: {city_dir}]")

# else:
#     raise ValueError(f"Unknown CLASSIFICATION_INPUT: {CLASSIFICATION_INPUT!r}. "
#                      "Use 'valid_only' or 'all_extracted'.")

# total = sum(len(v) for v in all_valid.values())
# print(f"\nall_valid populated: {len(all_valid)} cities, {total} total policies  "
#       f"(mode: {CLASSIFICATION_INPUT})")


### Step 8 — Guide-Integrated Classification

Classifies all valid policies (individual + sub rows from `all_valid`) using a **guide-integrated three-stage mechanism registry**:

1. **Stage 1 — Mechanism Extraction** (parallelized, 1 call/policy): strips city/numbers/dates, extracts a canonical `<action> → <climate_effect>` string per policy.  
2. **Stage 2 — Mechanism Classification** (parallelized, 1 call per *unique* mechanism): classifies the abstract mechanism *once* to lock the **primary category** for all structurally equivalent policies.  
3. **Stage 3 — Policy Finalization** (parallelized, 1 call/policy): uses policy text and verbatim source text to assign the expert secondary typology code, derive `secondary_categories`, and add instance-specific metadata (`instrument_type`, `instrument_directness`, `climate_relevance`, `key_indicators`, `co_benefits`).

Fields added per policy: `canonical_mechanism`, `mechanism_description`, `primary_category`, `secondary_categories`,  
`typology_code`, `typology_confidence`, `typology_evidence_quote`, `primary_causal_pathway`,  
`causal_mechanism_detail`, `dominant_pathway_test`, `mechanism_classification_reasoning`, `mechanism_confidence`,  
`instrument_type`, `instrument_directness`, `climate_relevance`, `key_indicators`, `co_benefits`,  
`classification_schema_version`, `secondary_profile`.

Saves `classified_policies.csv` and `trace_classification.json` under `outputs/{City}/`.

> **Vulnerability context**: populate `LOCATION_VULNERABILITIES` in `utils/consistent_classification.py` to help interpret policy context, but expert secondary assignment still requires explicit text evidence.

In [ ]:
# Step 8: Optimized 3-Stage Guide-Integrated Classification

import json

from utils.consistent_classification import (
    ConsistentPolicyClassifier,
    build_vulnerability_context,
)
from dspy.utils.parallelizer import ParallelExecutor
from collections import defaultdict as _defdict

classifier = ConsistentPolicyClassifier()
all_classified = {}

# ── Pool valid policies from all cities ──────────────────────────────────────
all_records = []
skipped_keys = []
_vuln_cache = {}

for entry in BATCH:
    key = city_key(entry["metadata"])
    df_valid = all_valid.get(key)
    if df_valid is None or df_valid.empty:
        print(f"[{key}] No valid policies — skipping classification.")
        all_classified[key] = pd.DataFrame()
        skipped_keys.append(key)
        continue

    if key not in _vuln_cache:
        _vuln_cache[key] = build_vulnerability_context(key)

    vuln_ctx = _vuln_cache[key]
    for rec in df_valid.to_dict(orient="records"):
        rec["__city_key"] = key
        rec["location"] = key
        rec["location_vulnerability_context"] = vuln_ctx
        all_records.append(rec)

active_keys = [city_key(e["metadata"]) for e in BATCH if city_key(e["metadata"]) not in skipped_keys]
print(f"\nPooled {len(all_records)} valid policies across {len(active_keys)} cities for classification.")

# ── Per-city input breakdown ──────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("INPUT BREAKDOWN BY CITY")
print(f"{'─'*60}")
_by_city = _defdict(list)
for rec in all_records:
    _by_city[rec["__city_key"]].append(rec)
for _k, _recs in _by_city.items():
    _roles = _defdict(int)
    for r in _recs:
        _roles[r.get("role", "unknown")] += 1
    _role_str = ", ".join(f"{v} {k}" for k, v in sorted(_roles.items()))
    print(f"  {_k:<20s} {len(_recs):>4d} policies  [{_role_str}]")
    for r in _recs:
        stmt = str(r.get("policy_statement", ""))[:90]
        print(f"    [{r.get('role','?'):10s}] {stmt}{'…' if len(str(r.get('policy_statement',''))) > 90 else ''}")

if not all_records:
    print("Nothing to classify.")
else:
    _N = max(NUM_THREADS, min(32, len(all_records)))

    # ── STAGE 1: Mechanism Extraction (parallelized) ──────────────────────────
    print(f"\n{'='*60}")
    print(f"STAGE 1: Extracting mechanisms from {len(all_records)} policies ({_N} threads) ...")
    print(f"{'='*60}")

    def _extract_one(record):
        extraction = classifier.extract_mechanism(
            policy_statement=record["policy_statement"],
            verbatim_text=record["verbatim_text"],
        )
        return {
            **record,
            "canonical_mechanism":   extraction.canonical_mechanism,
            "sector":                extraction.sector,
            "mechanism_description": extraction.mechanism_description,
        }

    _exec1 = ParallelExecutor(
        num_threads=_N,
        max_errors=len(all_records),
        provide_traceback=True,
    )
    _s1_raw = _exec1.execute(_extract_one, all_records)

    s1_records, s1_fail = [], 0
    for i, res in enumerate(_s1_raw):
        if res is None or isinstance(res, Exception):
            s1_fail += 1
            print(f"  ⚠ Stage 1 record {i} failed: {res}")
            fallback = dict(all_records[i])
            fallback["canonical_mechanism"]   = "unknown_mechanism → unknown_effect"
            fallback["sector"]                = "cross_sector"
            fallback["mechanism_description"] = "mechanism extraction failed"
            s1_records.append(fallback)
        else:
            s1_records.append(res)

    print(f"Stage 1 complete: {len(s1_records) - s1_fail} succeeded, {s1_fail} failed")

    # ── Stage 1 results: show what each policy mapped to ─────────────────────
    print(f"\n{'─'*60}")
    print("STAGE 1 RESULTS — policy → canonical_mechanism")
    print(f"{'─'*60}")
    for rec in s1_records:
        stmt  = str(rec.get("policy_statement", ""))[:70]
        mech  = rec.get("canonical_mechanism", "—")
        city  = rec.get("__city_key", "?")
        role  = rec.get("role", "?")
        print(f"  [{city:<15s} | {role:<10s}] {stmt:<72s}")
        print(f"    → {mech}")

    # ── STAGE 1.5: Fuzzy mechanism canonicalization ───────────────────────────
    # Merges near-identical mechanism strings (e.g. arrow/whitespace variants)
    # into a single canonical form before the registry is built. Sets
    # _mechanism_key on each record for consistent Stage 2 grouping.
    print(f"\n{'─'*60}")
    print("STAGE 1.5: Fuzzy mechanism clustering ...")
    print(f"{'─'*60}")
    s1_records = classifier.canonicalize_mechanisms(s1_records, threshold=0.85)

    # ── STAGE 2: Mechanism Classification (parallelized by unique mechanism) ──
    print(f"\n{'='*60}")
    print("STAGE 2: Classifying mechanisms for primary labels ...")
    print(f"{'='*60}")

    _mech_groups = _defdict(list)
    for rec in s1_records:
        _mech_groups[rec.get("_mechanism_key", rec["canonical_mechanism"])].append(rec)

    unique_mechs = list(_mech_groups.keys())
    print(f"  {len(unique_mechs)} unique mechanisms from {len(s1_records)} policies "
          f"({len(s1_records) - len(unique_mechs)} LLM calls saved vs row-by-row)")

    print(f"\n{'─'*60}")
    print("STAGE 2 GROUPS — mechanism → policies that share it")
    print(f"{'─'*60}")
    for mech, recs in sorted(_mech_groups.items(), key=lambda x: -len(x[1])):
        cities = list({r["__city_key"] for r in recs})
        print(f"\n  [{len(recs):>3d} policies | {', '.join(cities)}]  {mech}")
        for r in recs:
            stmt = str(r.get("policy_statement", ""))[:80]
            print(f"      · {stmt}{'…' if len(str(r.get('policy_statement',''))) > 80 else ''}")

    def _classify_mechanism(mech_key):
        group = _mech_groups[mech_key]
        reps = []
        seen_locations = set()
        for p in group:
            loc = p.get("location", p.get("__city_key", "unknown"))
            if loc in seen_locations or len(reps) >= 3:
                continue
            reps.append({
                "location": loc,
                "policy_statement": p.get("policy_statement", ""),
            })
            seen_locations.add(loc)

        return mech_key, classifier.classify_mechanism(
            canonical_mechanism=mech_key,
            sector=group[0].get("sector", "cross_sector"),
            mechanism_description=group[0].get("mechanism_description", ""),
            representative_policies=json.dumps(reps),
        )

    _exec2 = ParallelExecutor(
        num_threads=min(_N, len(unique_mechs)),
        max_errors=len(unique_mechs),
        provide_traceback=True,
    )

    def _safe_get(obj, key, default="—"):
        try:
            return obj[key]
        except Exception:
            if hasattr(obj, "get"):
                try:
                    return obj.get(key, default)
                except Exception:
                    return default
            return default

    def _augment_labels(mech_key, labels):
        recs = _mech_groups.get(mech_key, [])
        cities = sorted({r.get("__city_key", "?") for r in recs})

        base = {}
        if isinstance(labels, dict):
            base = labels
        elif hasattr(labels, "keys"):
            for k in labels.keys():
                try:
                    base[k] = labels[k]
                except Exception:
                    pass

        base = dict(base)
        base.setdefault("secondary_categories", "None")
        base.setdefault("policy_count", len(recs))
        base.setdefault("locations", ", ".join(cities))
        return base

    try:
        _s2_raw = _exec2.execute(_classify_mechanism, unique_mechs)
        registry = {}
        for res in _s2_raw:
            if res is None or isinstance(res, Exception):
                print(f"  ⚠ Stage 2 mechanism failed: {res}")
            else:
                mech_key, labels = res
                registry[mech_key] = _augment_labels(mech_key, labels)
        classifier.mechanism_registry = registry
    except (AttributeError, TypeError):
        registry = classifier.stage2_classify_mechanisms(s1_records)
        registry = {k: _augment_labels(k, v) for k, v in registry.items()}
        classifier.mechanism_registry = registry

    print(f"\nMechanism registry: {len(registry)} unique mechanisms")
    for mech, info in registry.items():
        print(f"\n  {mech}")
        print(f"    Primary:   {_safe_get(info, 'primary_category', '—')}")
        print(f"    Policies:  {_safe_get(info, 'policy_count', '?')} across {_safe_get(info, 'locations', '?')}")

    # ── Persist mechanism registry with representative policies ─────────────────
    def _repr_policy(r):
        return {
            "policy_statement": str(r.get("policy_statement", ""))[:500],
            "verbatim_text": str(r.get("verbatim_text", ""))[:500],
            "city": r.get("__city_key", "?"),
            "role": r.get("role", "?"),
        }
    def _json_val(v):
        if v is None or isinstance(v, (str, int, float, bool)):
            return v
        return str(v)
    registry_trace = []
    for mech_key, info in registry.items():
        reps = _mech_groups.get(mech_key, [])[:3]
        entry = {"canonical_mechanism": mech_key}
        for k, v in info.items():
            try:
                entry[k] = _json_val(v)
            except Exception:
                entry[k] = str(v) if v is not None else None
        entry["representative_policies"] = [_repr_policy(r) for r in reps]
        entry["all_policy_statements"] = [str(r.get("policy_statement", ""))[:200] for r in _mech_groups.get(mech_key, [])]
        registry_trace.append(entry)
    registry_path = OUTPUT_DIR / "mechanism_registry.json"
    with open(registry_path, "w", encoding="utf-8") as f:
        json.dump(registry_trace, f, ensure_ascii=False, indent=2)
    print(f"\n  → {registry_path}")

    # ── STAGE 3: Policy-level secondary typing + enrichment ──────────────────
    print(f"\n{'='*60}")
    print(f"STAGE 3: Finalizing {len(s1_records)} policies ({_N} threads) ...")
    print(f"{'='*60}")

    _registry_snapshot = dict(classifier.mechanism_registry)

    def _finalize_one(record):
        mech_key = record.get("_mechanism_key", record["canonical_mechanism"])
        mech_labels = _registry_snapshot[mech_key]
        location_ctx = record.get("location_vulnerability_context", "No vulnerability context provided")

        secondary = classifier.classify_policy_secondary(
            primary_category=mech_labels["primary_category"],
            policy_statement=record["policy_statement"],
            verbatim_text=record["verbatim_text"],
            canonical_mechanism=record["canonical_mechanism"],
            mechanism_description=record["mechanism_description"],
            location_vulnerability_context=location_ctx,
        )

        enrichment = classifier.enrich_policy(
            policy_statement=record["policy_statement"],
            verbatim_text=record["verbatim_text"],
            assigned_primary=mech_labels["primary_category"],
            assigned_causal_pathway=mech_labels["primary_causal_pathway"],
            assigned_causal_detail=mech_labels["causal_mechanism_detail"],
            assigned_typical_instruments=mech_labels["typical_policy_instruments"],
            location_vulnerability_context=location_ctx,
        )

        return {
            **record,
            **secondary,
            "primary_category":                   mech_labels["primary_category"],
            "primary_causal_pathway":             mech_labels["primary_causal_pathway"],
            "causal_mechanism_detail":            mech_labels["causal_mechanism_detail"],
            "dominant_pathway_test":              mech_labels["dominant_pathway_test"],
            "mechanism_classification_reasoning": mech_labels["classification_reasoning"],
            "mechanism_confidence":               mech_labels["confidence_score"],
            "mechanism_edge_case_notes":          mech_labels["edge_case_notes"],
            "instrument_type":                    enrichment.instrument_type,
            "instrument_directness":              enrichment.instrument_directness,
            "climate_relevance":                  enrichment.climate_relevance,
            "key_indicators":                     enrichment.key_indicators,
            "co_benefits":                        enrichment.co_benefits,
            "instance_edge_case_notes":           enrichment.edge_case_notes,
        }

    _exec3 = ParallelExecutor(
        num_threads=_N,
        max_errors=len(s1_records),
        provide_traceback=True,
    )
    _s3_raw = _exec3.execute(_finalize_one, s1_records)

    classified_flat, s3_fail = [], 0
    for i, res in enumerate(_s3_raw):
        if res is None or isinstance(res, Exception):
            s3_fail += 1
            print(f"  ⚠ Stage 3 record {i} failed: {res}")
            classified_flat.append({
                **s1_records[i],
                "secondary_categories": "None",
                "typology_code": "None",
                "typology_confidence": 0.0,
                "typology_evidence_quote": "None",
                "classification_schema_version": "guide_secondary_typology_v1",
                "secondary_profile": "policy_text_evidence_gated",
                "instrument_type": "ERROR",
                "climate_relevance": "ERROR",
            })
        else:
            classified_flat.append(res)

    print(f"\nStage 3 complete: {len(classified_flat) - s3_fail} succeeded, {s3_fail} failed")

    _per_city = _defdict(list)
    for rec in classified_flat:
        _per_city[rec["__city_key"]].append(rec)

    _TRACE_COLS = [
        "policy_statement", "role",
        "canonical_mechanism", "mechanism_description", "sector",
        "primary_category", "secondary_categories",
        "typology_code", "typology_confidence", "typology_evidence_quote",
        "primary_causal_pathway", "causal_mechanism_detail", "dominant_pathway_test",
        "mechanism_classification_reasoning", "mechanism_confidence", "mechanism_edge_case_notes",
        "instrument_type", "instrument_directness", "climate_relevance",
        "key_indicators", "co_benefits", "instance_edge_case_notes",
        "classification_schema_version", "secondary_profile",
    ]

    dfs_to_write = {}
    for entry in BATCH:
        key = city_key(entry["metadata"])
        if key not in _per_city:
            continue
        df_classified = pd.DataFrame(_per_city[key])
        df_classified.drop(columns=["__city_key", "_mechanism_key"], errors="ignore", inplace=True)
        all_classified[key] = df_classified
        dfs_to_write[key] = df_classified

    for key, df_classified in dfs_to_write.items():
        city_output_dir = OUTPUT_DIR / key
        city_output_dir.mkdir(parents=True, exist_ok=True)

        out_csv = city_output_dir / "classified_policies.csv"
        df_classified.to_csv(out_csv, index=False)

        trace_cols = [c for c in _TRACE_COLS if c in df_classified.columns]
        trace_data = df_classified[trace_cols].to_dict(orient="records")
        out_traces = city_output_dir / "trace_classification.json"
        with open(out_traces, "w", encoding="utf-8") as f:
            json.dump(trace_data, f, ensure_ascii=False, indent=2)

        cat_counts = df_classified["primary_category"].value_counts()
        _conf_col  = df_classified.get("mechanism_confidence", pd.Series(dtype=float))
        avg_conf   = _conf_col.mean()
        n_multi    = df_classified["secondary_categories"].apply(
            lambda x: bool(x) and str(x).strip().lower() not in ("none", "")
        ).sum()

        print(f"\n[{key}] ── Classification Summary ──")
        print(f"  Total classified : {len(df_classified)}")
        print(f"  Multi-label      : {n_multi}")
        if pd.notna(avg_conf):
            print(f"  Avg confidence   : {avg_conf:.3f}")
        print(f"  Category distribution:")
        for cat, cnt in cat_counts.items():
            pct = cnt / len(df_classified) * 100
            print(f"    {cat:<30s} {cnt:>4d}  ({pct:.1f}%)")
        print(f"  → {out_csv}")
        print(f"  → {out_traces}")

    print(f"\n{'='*60}")
    print(f"REGISTRY SUMMARY")
    print(f"  Total policies classified : {len(all_records)}")
    print(f"  Unique mechanisms found   : {len(registry)}")
    print(f"  Stage 2 LLM calls saved   : {len(all_records) - len(registry)} "
          f"(vs. row-by-row baseline)")
    print(f"{'='*60}")

[Geneva] No valid policies — skipping classification.

Pooled 364 valid policies across 9 cities for classification.

────────────────────────────────────────────────────────────
INPUT BREAKDOWN BY CITY
────────────────────────────────────────────────────────────
  Chicago                62 policies  [2 individual, 13 parent, 47 sub]
    [parent    ] Strategy 1 (Pillar 1): Retrofit buildings.
    [sub       ] Retrofit residential buildings with 4 or fewer units: retrofit 20% by 2030 and 50% by 2040…
    [sub       ] Retrofit 20% of total 5+ unit residential buildings by 2030.
    [sub       ] Retrofit 20% of total industrial buildings by 2030.
    [sub       ] Retrofit 90% of total City-owned and sister agency-owned buildings by 2035.
    [parent    ] Strategy 2 (Pillar 1): Connect communities to renewable energy.
    [sub       ] Install 5 megawatts of co-owned community solar projects by 2025.
    [sub       ] Increase Chicago-based community renewables to 20 megawatts by 2025.
    [

2026/03/30 19:10:56 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['canonical_mechanism', 'representative_policies']. Missing: ['sector', 'mechanism_description'].
2026/03/30 19:10:56 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['canonical_mechanism', 'representative_policies']. Missing: ['sector', 'mechanism_description'].
2026/03/30 19:10:56 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['canonical_mechanism', 'representative_policies']. Missing: ['sector', 'mechanism_description'].
2026/03/30 19:10:56 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['canonical_mechanism', 'representative_policies']. Missing: ['sector', 'mechanism_description'].


  [mechanism-cluster] merged 16 variant mechanism strings into 195 canonical forms

STAGE 2: Classifying mechanisms ...
  195 unique mechanisms from 364 policies (169 LLM calls saved vs row-by-row)

────────────────────────────────────────────────────────────
STAGE 2 GROUPS — mechanism → policies that share it
────────────────────────────────────────────────────────────

  [ 21 policies | Chicago, Portugal, Kuwait, Hiroshima, Seattle, Dakar]  environmental_monitoring_system_→_adaptation_planning_support
      · Strategy 1 (Pillar 5): Collect relevant data.
      · Develop a water and soil quality measurement and mitigation strategy by 2023.
      · Establish a robust outdoor air quality monitoring network by 2025.
      · Strategy 2 (Pillar 5): Enable data-driven decision-making.
      · Publish citywide and community-level quality-of-life metrics on equity and susta…
      · Near-term climate action priorities to assess GHG emissions impacts of City deci…
      · Dakar will pursue ada

2026/03/30 19:10:56 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['canonical_mechanism', 'representative_policies']. Missing: ['sector', 'mechanism_description'].
2026/03/30 19:10:56 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['canonical_mechanism', 'representative_policies']. Missing: ['sector', 'mechanism_description'].


Processed 1 / 195 examples:   0%|          | 0/195 [00:00<?, ?it/s]

...

Processed 192 / 195 examples:  98%|█████████▊| 191/195 [00:01<00:00, 74.65it/s]

2026/03/30 19:10:58 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['canonical_mechanism', 'representative_policies']. Missing: ['sector', 'mechanism_description'].


Processed 192 / 195 examples:  98%|█████████▊| 192/195 [00:02<00:00, 133.56it/s]

2026/03/30 19:10:58 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['canonical_mechanism', 'representative_policies']. Missing: ['sector', 'mechanism_description'].


Processed 195 / 195 examples: 100%|██████████| 195/195 [00:02<00:00, 96.01it/s] 

Mechanism registry: 195 unique mechanisms

  building_retrofit_→_energy_use_reduction
    Primary:   Mitigation
    Secondary: None
    Policies:  13 across Chicago, Hiroshima, Miami_Dade, Seattle

  renewable_energy_access_program_→_fossil_generation_displacement
    Primary:   Mitigation
    Secondary: None
    Policies:  2 across Chicago, Miami_Dade

  solar_pv_deployment_→_fossil_generation_displacement
    Primary:   Mitigation
    Secondary: None
    Policies:  7 across Chicago, Miami_Dade

  renewable_energy_target_→_grid_decarbonization
    Primary:   Mitigation
    Secondary: None
    Policies:  6 across Chicago, Dakar, Miami_Dade

  community_renewables_subscription_expansion_→_fossil_generation_displacement
    Primary:   Mitigation
    Secondary: None
    Policies:  1 across Chicago

  waste_diversion_→_landfill_methane_avoidance
    Primary:   Mitigation
    Secondary: None
    Policies:  12 

In [ ]:
# ── Post-Step 8: Export final CSV + trace lookup + per-policy trace files ──
#
# For every city that has both a classified_policies.csv and combined_policies.csv
# this step rewrites the final presentation outputs to:
#   1. `combined_policies.csv`       -> minimal final table for presentation
#   2. `policy_trace_lookup.csv`     -> tells you how to find validator traces
#   3. `policy_traces/*.json`        -> one classification trace per policy row

import importlib

from utils import exports as exports_mod

exports_mod = importlib.reload(exports_mod)
export_final_combined_with_traces = exports_mod.export_final_combined_with_traces

for entry in BATCH:
    key = city_key(entry["metadata"])
    city_dir = OUTPUT_DIR / key
    classified_path = city_dir / "classified_policies.csv"
    combined_path = city_dir / "combined_policies.csv"

    if not classified_path.exists():
        print(f"[{key}] classified_policies.csv not found; skipping.")
        continue

    if not combined_path.exists():
        print(f"[{key}] combined_policies.csv not found; skipping.")
        continue

    df_cls = pd.read_csv(classified_path)
    df_comb = pd.read_csv(combined_path)

    paths = export_final_combined_with_traces(
        combined=df_comb,
        df_classified=df_cls,
        output_dir=city_dir,
    )

    refreshed = pd.read_csv(paths["combined_policies"])
    if "parent_row_type" in refreshed.columns:
        print(f"[{key}] final combined parent_row_type: {refreshed['parent_row_type'].notna().sum()}/{len(refreshed)}")
    print(f"[{key}] final combined → {paths['combined_policies']}")
    print(f"[{key}] trace lookup → {paths['policy_trace_lookup']}")
    print(f"[{key}] per-policy traces → {paths['policy_traces_dir']}")

print("\nDone.")


[Chicago] final combined parent_row_type: 0/62
[Chicago] final combined → outputs/Chicago/combined_policies.csv
[Chicago] trace lookup → outputs/Chicago/policy_trace_lookup.csv
[Chicago] per-policy traces → outputs/Chicago/policy_traces
[Seattle] final combined parent_row_type: 0/17
[Seattle] final combined → outputs/Seattle/combined_policies.csv
[Seattle] trace lookup → outputs/Seattle/policy_trace_lookup.csv
[Seattle] per-policy traces → outputs/Seattle/policy_traces
[Las_Vegas] final combined parent_row_type: 0/16
[Las_Vegas] final combined → outputs/Las_Vegas/combined_policies.csv
[Las_Vegas] trace lookup → outputs/Las_Vegas/policy_trace_lookup.csv
[Las_Vegas] per-policy traces → outputs/Las_Vegas/policy_traces
[Miami_Dade] final combined parent_row_type: 0/54
[Miami_Dade] final combined → outputs/Miami_Dade/combined_policies.csv
[Miami_Dade] trace lookup → outputs/Miami_Dade/policy_trace_lookup.csv
[Miami_Dade] per-policy traces → outputs/Miami_Dade/policy_traces
[Austin] final co

In [ ]:
import pandas as pd

valid_frames = []
classified_frames = []

for entry in BATCH:
    key = city_key(entry["metadata"])
    city_dir = OUTPUT_DIR / key

    combined_path = city_dir / "combined_policies.csv"
    if combined_path.exists():
        df_combined = pd.read_csv(combined_path)
        valid_frames.append(df_combined.assign(city=key))

    classified_path = city_dir / "classified_policies.csv"
    if classified_path.exists():
        df_classified = pd.read_csv(classified_path)
        classified_frames.append(df_classified.assign(city=key))

master_valid_df = pd.concat(valid_frames, ignore_index=True) if valid_frames else pd.DataFrame()
master_classified_df = pd.concat(classified_frames, ignore_index=True) if classified_frames else pd.DataFrame()

if not master_valid_df.empty and "parent_row_type" not in master_valid_df.columns:
    master_valid_df["parent_row_type"] = pd.NA

for key in master_valid_df.get("city", pd.Series(dtype=str)).dropna().unique() if not master_valid_df.empty else []:
    df_init = None
    for trace_name in ("trace_initiative_policies.csv", "trace_initiative_policies_valid.csv"):
        init_trace = OUTPUT_DIR / key / trace_name
        if not init_trace.exists():
            continue
        candidate = pd.read_csv(init_trace)
        if "parent_row_type" in candidate.columns and "parent_statement" in candidate.columns:
            df_init = candidate
            break

    if df_init is None:
        continue

    parent_map = (
        df_init.loc[
            df_init["parent_statement"].notna()
            & df_init["parent_row_type"].notna()
        ]
        .drop_duplicates(subset=["parent_statement"], keep="first")
        .set_index("parent_statement")["parent_row_type"]
    )
    if parent_map.empty:
        continue

    city_mask = master_valid_df["city"] == key
    parent_mask = city_mask & (master_valid_df["role"] == "parent")
    sub_mask = city_mask & (master_valid_df["role"] == "sub")

    parent_fill = master_valid_df.loc[parent_mask, "policy_statement"].map(parent_map)
    sub_fill = master_valid_df.loc[sub_mask, "parent_statement"].map(parent_map)

    master_valid_df.loc[parent_mask, "parent_row_type"] = (
        master_valid_df.loc[parent_mask, "parent_row_type"].fillna(parent_fill)
    )
    master_valid_df.loc[sub_mask, "parent_row_type"] = (
        master_valid_df.loc[sub_mask, "parent_row_type"].fillna(sub_fill)
    )

if not master_valid_df.empty:
    valid_cols = ["city"] + [c for c in master_valid_df.columns if c != "city"]
    master_valid_df = master_valid_df[valid_cols]
    master_valid_df.to_csv("outputs/all_cities_valid_policies.csv", index=False)

if not master_classified_df.empty:
    classified_cols = ["city"] + [c for c in master_classified_df.columns if c != "city"]
    master_classified_df = master_classified_df[classified_cols]
    master_classified_df.to_csv("outputs/all_cities_classified_policies.csv", index=False)

print(f"Combined {len(valid_frames)} city files into outputs/all_cities_valid_policies.csv")
print(f"Total valid policies: {len(master_valid_df)}")
print(f"Combined {len(classified_frames)} city files into outputs/all_cities_classified_policies.csv")
print(f"Total classified policies: {len(master_classified_df)}")
if not master_valid_df.empty and master_valid_df["parent_row_type"].notna().any():
    print(f"Rows with parent_row_type: {master_valid_df['parent_row_type'].notna().sum()}")
elif not master_valid_df.empty:
    print("parent_row_type still empty: rerun initiative validation so per-city trace files include the column.")


Combined 10 cities into a single file: outputs/all_cities_valid_policies.csv
Total valid policies: 364
Rows with parent_row_type: 65
